# Predicting AstraZeneca Stock Direction Using a Blending Ensemble

**Asset:** AstraZeneca PLC ADR (NASDAQ: AZN)  
**Frequency:** Daily  
**Task:** Binary classification of the next executable open-to-open return  
**Strategy:** Long AZN or cash

This notebook implements the empirical analysis supporting the CQF project report. It is organised to prevent look-ahead bias: data are kept in chronological order, transformations are fitted only on the relevant training window, and the final test set is left untouched until the model specification has been fixed.

## 1. Research design

### Target construction

### Target construction

At the close of day $t$, the model uses all information available up to that point to predict AZN's next tradable open-to-open return. Since the signal is generated after the market closes, the earliest possible entry is at the open of day $t+1$. The corresponding return is calculated as:

$$
r_{t+1}^{O-O} = \frac{O_{t+2}}{O_{t+1}} - 1
$$

Here, $O_{t+1}$ is the entry price and $O_{t+2}$ is the exit price.

The return is then converted into a binary target. A positive return is labelled 1, while a zero or negative return is labelled 0. The baseline analysis uses a return cutoff of zero. Alternative cutoffs are considered later as robustness tests.

This timing convention avoids assuming that a signal generated using the closing price on day $t$ could be executed retrospectively at the same closing price.

Adjusted OHLC prices are used to maintain consistency across stock splits and dividend adjustments. The target is therefore interpreted as an adjusted open-to-open return. Actual trades would take place at raw market prices, while adjusted prices provide a continuous series for modelling and backtesting.

## 2. Imports and reproducibility

If an import fails, install the packages listed in `requirements.txt` and restart the kernel.

In [ ]:
from pathlib import Path
import platform
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import yfinance as yf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"yfinance: {yf.__version__}")

## 3. Central configuration

The study uses five complete calendar years. AZN is the only prediction and trading target. Other instruments are explanatory variables only. The end date supplied to `yfinance` is exclusive, so `2025-01-01` includes observations through the end of 2024.

In [ ]:
TARGET = "AZN"
EXTERNAL_TICKERS = {
    "SPY": "SPY",       # broad US equity market
    "XLV": "XLV",       # US healthcare sector
    "IBB": "IBB",       # biotechnology sector
    "VIX": "^VIX",      # market risk sentiment
    "GBPUSD": "GBPUSD=X" # sterling / US dollar
}
START_DATE = "2021-01-01"
END_DATE = "2026-01-01"
RETURN_LABEL_CUTOFF = 0.0
TRAIN_FRACTION = 0.60
BLEND_FRACTION = 0.20
TEST_FRACTION = 0.20
ONE_WAY_COST_BPS = 5.0

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

assert np.isclose(TRAIN_FRACTION + BLEND_FRACTION + TEST_FRACTION, 1.0)
print(f"Study period: {START_DATE} to {END_DATE} (end date exclusive)")
print(f"Data directory: {DATA_DIR.resolve()}")

## 4. Download adjusted daily market data

`auto_adjust=True` applies Yahoo Finance's split and dividend adjustments consistently to OHLC prices. Each ticker is downloaded separately to avoid ambiguous multi-index columns. Raw downloads are cached as CSV files for reproducibility.

In [ ]:
def download_ohlcv(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Download adjusted daily OHLCV data and return a validated DataFrame."""
    frame = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,
        actions=False,
        progress=False,
        threads=False,
    )
    if frame.empty:
        raise ValueError(f"No observations downloaded for {ticker}.")
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame.index.name = "Date"
    frame = frame.loc[~frame.index.duplicated(keep="last")].sort_index()
    required = ["Open", "High", "Low", "Close", "Volume"]
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{ticker} is missing required columns: {missing}")
    return frame[required].astype(float)

ticker_map = {"AZN": TARGET, **EXTERNAL_TICKERS}
raw_data = {}
for name, ticker in ticker_map.items():
    raw_data[name] = download_ohlcv(ticker, START_DATE, END_DATE)
    raw_data[name].to_csv(DATA_DIR / f"{name}_adjusted_ohlcv.csv")

download_summary = pd.DataFrame({
    name: {
        "ticker": ticker_map[name],
        "start": frame.index.min().date(),
        "end": frame.index.max().date(),
        "observations": len(frame),
        "duplicate_dates": int(frame.index.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for name, frame in raw_data.items()
}).T
download_summary

## 5. Basic validation and alignment

AZN's trading calendar is the master calendar. External close series are reindexed to AZN dates. A forward fill is permitted only for external data because a market or FX series can be closed on an AZN trading day; it uses the most recently available historical observation and never a future observation. Long gaps are reported for inspection.

In [ ]:
def validate_ohlcv(frame: pd.DataFrame, name: str) -> None:
    """Run basic OHLCV integrity checks and raise on invalid prices."""
    if not frame.index.is_monotonic_increasing:
        raise ValueError(f"{name}: dates are not increasing.")
    if frame.index.has_duplicates:
        raise ValueError(f"{name}: duplicate dates remain.")
    if (frame[["Open", "High", "Low", "Close"]] <= 0).any().any():
        raise ValueError(f"{name}: non-positive price detected.")
    if (frame["Volume"] < 0).any():
        raise ValueError(f"{name}: negative volume detected.")

for name, frame in raw_data.items():
    validate_ohlcv(frame, name)

azn = raw_data["AZN"].copy()
aligned = azn.add_prefix("AZN_")
alignment_report = []

for name in EXTERNAL_TICKERS:
    series = raw_data[name]["Close"].reindex(azn.index)
    missing_before = int(series.isna().sum())
    series = series.ffill(limit=5)
    missing_after = int(series.isna().sum())
    aligned[f"{name}_Close"] = series
    alignment_report.append({
        "series": name,
        "missing_before_ffill": missing_before,
        "missing_after_ffill": missing_after,
    })

alignment_report = pd.DataFrame(alignment_report).set_index("series")
alignment_report

## 6. Construct the executable target

The target is aligned to the feature date. On row \(t\), `target_return` is the future adjusted open-to-open return beginning at the next available AZN open. The final two rows naturally have unavailable targets and are removed only from the modelling sample.

In [ ]:
model_data = aligned.copy()
model_data["target_return"] = (
    model_data["AZN_Open"].shift(-2) / model_data["AZN_Open"].shift(-1) - 1.0
)
model_data["target"] = np.where(
    model_data["target_return"].notna(),
    (model_data["target_return"] > RETURN_LABEL_CUTOFF).astype(int),
    np.nan,
)

labelled_data = model_data.dropna(subset=["target_return", "target"]).copy()
labelled_data["target"] = labelled_data["target"].astype(int)

target_summary = pd.Series({
    "observations": len(labelled_data),
    "positive_days": int(labelled_data["target"].sum()),
    "negative_days": int((labelled_data["target"] == 0).sum()),
    "positive_rate": labelled_data["target"].mean(),
    "mean_target_return": labelled_data["target_return"].mean(),
    "daily_target_volatility": labelled_data["target_return"].std(),
})
target_summary.to_frame("value")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
azn["Close"].plot(ax=axes[0], color="#1f77b4", linewidth=1.4)
axes[0].set_title("AZN adjusted closing price")
axes[0].set_ylabel("USD")

labelled_data["target"].value_counts().sort_index().plot.bar(
    ax=axes[1], color=["#d95f02", "#1b9e77"]
)
axes[1].set_title("Binary target class counts")
axes[1].set_xlabel("Class (0 = non-positive, 1 = positive)")
axes[1].set_ylabel("Number of observations")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### Stage 1 checkpoint
The data download and initial validation were completed successfully. AZN contains approximately 1,250 daily observations, with no duplicate dates, missing values, or invalid prices. The external series were aligned to the AZN trading calendar, and no missing values remained after forward filling. 

The two target classes were also reasonably balanced, with positive returns accounting for approximately 51% of the observations. Based on these checks, the dataset is suitable for the next stage of feature engineering and training-period exploratory analysis.

# Stage 2 - Candidate features, chronological split, and training-only EDA

All predictors use information available at or before the close of date \(t\); no feature uses a negative shift. Established technical indicators are called from the `ta` library. Deterministic rolling features may be calculated before splitting, but all fitted preprocessing and feature selection will be estimated on the training sample only.

SPY, XLV, IBB and VIX share the US trading session with AZN, so date-\(t\) closes are treated as available after the session. Yahoo's GBP/USD daily timestamp is only an approximate AZN-close snapshot; this limitation will be stated in the report.

In [ ]:
from src.features import build_candidate_features

PURGE_GAP = 2
candidate_features = build_candidate_features(model_data, list(EXTERNAL_TICKERS))
feature_names = candidate_features.columns.tolist()
dataset = pd.concat([candidate_features, model_data[["target_return", "target"]]], axis=1)
dataset = dataset.dropna(subset=feature_names + ["target_return", "target"]).copy()
dataset["target"] = dataset["target"].astype(int)
pd.Series({"candidate_features": len(feature_names), "complete_rows": len(dataset),
           "first_date": dataset.index.min().date(), "last_date": dataset.index.max().date(),
           "missing_cells": int(dataset.isna().sum().sum())}).to_frame("value")

## Chronological split with purging

The label on \(t\) uses the opens on \(t+1\) and \(t+2\). Two rows are therefore removed between adjacent samples so that a training label cannot reach into the blending period and a blending label cannot reach into the final test period.

In [ ]:
n = len(dataset)
usable_n = n - 2 * PURGE_GAP
n_train = int(np.floor(usable_n * TRAIN_FRACTION))
n_blend = int(np.floor(usable_n * BLEND_FRACTION))
train_end = n_train
blend_start, blend_end = train_end + PURGE_GAP, train_end + PURGE_GAP + n_blend
test_start = blend_end + PURGE_GAP
train_df = dataset.iloc[:train_end].copy()
blend_df = dataset.iloc[blend_start:blend_end].copy()
test_df = dataset.iloc[test_start:].copy()
assert len(train_df) + len(blend_df) + len(test_df) + 2 * PURGE_GAP == n
assert train_df.index.max() < blend_df.index.min() < test_df.index.min()
split_summary = pd.DataFrame({
    "Training": [train_df.index.min().date(), train_df.index.max().date(), len(train_df), train_df.target.mean()],
    "Blending": [blend_df.index.min().date(), blend_df.index.max().date(), len(blend_df), blend_df.target.mean()],
    "Final test": [test_df.index.min().date(), test_df.index.max().date(), len(test_df), test_df.target.mean()]},
    index=["start", "end", "observations", "positive_rate"]).T
split_summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
model_data.loc[dataset.index, "AZN_Close"].plot(ax=ax, color="0.55", linewidth=1.1)
for frame, color, label in [(train_df, "#1f77b4", "Training"), (blend_df, "#ff7f0e", "Blending"), (test_df, "#2ca02c", "Final test")]:
    ax.axvspan(frame.index.min(), frame.index.max(), alpha=0.17, color=color, label=label)
ax.set(title="Chronological model-development partitions", ylabel="AZN adjusted close (USD)")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## Training-only exploratory analysis

The following diagnostics use `train_df` only. The blending and final-test distributions remain unseen during feature screening.

In [ ]:
X_train_raw, y_train = train_df[feature_names], train_df["target"]
eda_summary = X_train_raw.agg(["mean", "std", "min", "median", "max"]).T
eda_summary["skew"] = X_train_raw.skew()
eda_summary["unique_values"] = X_train_raw.nunique()
eda_summary.sort_values("skew", key=lambda s: s.abs(), ascending=False).head(15)

In [ ]:
plot_features = ["azn_return_1d", "azn_return_5d", "azn_rsi_14", "azn_realized_vol_20", "azn_volume_z_20", "vix_return_1d"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, plot_features):
    sns.histplot(data=train_df, x=col, hue="target", stat="density", common_norm=False, element="step", fill=False, bins=30, ax=ax)
    ax.set_title(col.replace("_", " ").title())
plt.suptitle("Training-period feature distributions by future direction", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
train_corr = X_train_raw.corr(method="spearman")
upper = train_corr.abs().where(np.triu(np.ones(train_corr.shape), k=1).astype(bool))
high_corr_pairs = upper.stack().rename("abs_spearman_corr").sort_values(ascending=False).reset_index()
high_corr_pairs.columns = ["feature_1", "feature_2", "abs_spearman_corr"]
print(f"Pairs with |Spearman correlation| >= 0.90: {(high_corr_pairs.abs_spearman_corr >= 0.90).sum()}")
high_corr_pairs.head(20)

In [ ]:
corr_target = train_df[feature_names + ["target"]].corr(method="spearman")["target"].drop("target")
top_target_features = corr_target.sort_values(key=lambda s: s.abs(), ascending=False).head(15)
ordered = top_target_features.sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
ordered.plot.barh(ax=ax, color=np.where(ordered >= 0, "#1b9e77", "#d95f02"))
ax.set(title="Largest feature-target Spearman correlations (training only)", xlabel="Spearman correlation")
plt.tight_layout(); plt.show()
top_target_features.to_frame("spearman_with_target")

### Stage 2 checkpoint

Confirm that feature construction leaves no missing values and that the three samples remain chronological. Strongly correlated momentum and volatility groups are expected. Stage 3 will perform training-only correlation grouping, mutual-information screening and preprocessing before model tuning.

# Stage 3 - Leakage-safe preprocessing and feature reduction

Three operations are used. First, extreme values are clipped to the 0.5% and 99.5% quantiles learned from the relevant training window. Second, mutual information ranks potentially nonlinear associations with the label. Third, a greedy correlation filter retains the highest-ranked member of each group whose absolute Spearman correlation exceeds 0.90. A maximum of 30 predictors is retained for the diagnostic specification.

These are implemented as scikit-learn transformers. During model tuning the transformers will sit inside each pipeline, so their quantiles, mutual-information scores and retained features are re-estimated separately inside every time-series training fold.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.preprocessing import QuantileClipper, CorrelationMISelector

X_train = train_df[feature_names].copy()
y_train = train_df["target"].copy()
X_blend = blend_df[feature_names].copy()
y_blend = blend_df["target"].copy()
X_test = test_df[feature_names].copy()
y_test = test_df["target"].copy()

tscv = TimeSeriesSplit(n_splits=5, test_size=80, gap=PURGE_GAP)


In [ ]:
cv_rows = []
for fold, (fit_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    cv_rows.append({
        "fold": fold, "fit_start": X_train.index[fit_idx[0]].date(),
        "fit_end": X_train.index[fit_idx[-1]].date(), "fit_n": len(fit_idx),
        "validation_start": X_train.index[val_idx[0]].date(),
        "validation_end": X_train.index[val_idx[-1]].date(), "validation_n": len(val_idx),
        "gap_rows": val_idx[0] - fit_idx[-1] - 1})
cv_summary = pd.DataFrame(cv_rows).set_index("fold")
cv_summary

In [ ]:
diagnostic_clipper = QuantileClipper(lower=0.005, upper=0.995)
X_train_clipped = diagnostic_clipper.fit_transform(X_train)
diagnostic_selector = CorrelationMISelector(max_features=30, corr_threshold=0.90, random_state=SEED)
diagnostic_selector.fit(X_train_clipped, y_train)
selected_features_30 = diagnostic_selector.selected_features_
selection_table = pd.DataFrame({
    "feature": selected_features_30,
    "mutual_information": diagnostic_selector.mi_scores_.loc[selected_features_30].values
}).set_index("feature")
print(f"Selected {len(selected_features_30)} of {len(feature_names)} candidate features.")
selection_table

In [ ]:
top_mi = diagnostic_selector.mi_scores_.head(20).sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
top_mi.plot.barh(ax=ax, color="#4c78a8")
ax.set(title="Top mutual-information scores (base training sample only)", xlabel="Estimated mutual information")
plt.tight_layout(); plt.show()

## Feature-selection stability across expanding-window folds

A feature selected in many folds is more credible than one selected only in the longest training window. This diagnostic does not force the final model to use only stable features; it measures how sensitive the data-driven selection is to the estimation period.

In [ ]:
selection_counts = pd.Series(0, index=feature_names, dtype=int)
fold_selected_features = {}
for fold, (fit_idx, _) in enumerate(tscv.split(X_train), start=1):
    fold_clipper = QuantileClipper(lower=0.005, upper=0.995)
    X_fold = fold_clipper.fit_transform(X_train.iloc[fit_idx])
    fold_selector = CorrelationMISelector(max_features=30, corr_threshold=0.90, random_state=SEED)
    fold_selector.fit(X_fold, y_train.iloc[fit_idx])
    fold_selected_features[fold] = fold_selector.selected_features_
    selection_counts.loc[fold_selector.selected_features_] += 1
selection_stability = selection_counts.sort_values(ascending=False).to_frame("folds_selected")
selection_stability["selection_rate"] = selection_stability["folds_selected"] / tscv.n_splits
selection_stability.head(25)

In [ ]:
# Template used later for Logistic Regression; tree pipelines omit StandardScaler.
linear_preprocessing_template = Pipeline([
    ("clipper", QuantileClipper(lower=0.005, upper=0.995)),
    ("selector", CorrelationMISelector(max_features=30, corr_threshold=0.90, random_state=SEED)),
    ("scaler", StandardScaler()),
])
print(linear_preprocessing_template)

### Stage 3 checkpoint

Inspect the five expanding-window folds, selected-feature table and stability frequencies. Zero mutual-information estimates are valid finite-sample estimates and should not be manually changed. Stage 4 will place these transformers inside Logistic Regression, Random Forest and XGBoost pipelines and optimise their hyperparameters using ROC-AUC and balanced accuracy within the five training folds.

# Stage 4 - Base learners and time-series hyperparameter optimisation

The three base learners have deliberately different inductive biases: regularised Logistic Regression is linear and low variance; Random Forest captures nonlinear threshold effects through bagged trees; XGBoost fits regularised sequential trees. Each complete pipeline is optimised exclusively inside the 60% base-training sample using the five expanding-window folds defined above.

ROC-AUC is the refit criterion because it evaluates probability ranking without fixing a trading cutoff. Balanced accuracy and negative log loss are recorded as secondary criteria. The blending sample is not used to choose base-model hyperparameters; it is used afterwards to generate inputs for the meta-model.

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, brier_score_loss, log_loss)
from xgboost import XGBClassifier

SCORING = {
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss",
}


In [ ]:
def make_selector():
    return CorrelationMISelector(max_features=30, corr_threshold=0.90, random_state=SEED)

base_pipelines = {
    "Logistic Regression": Pipeline([
        ("clipper", QuantileClipper()), ("selector", make_selector()),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(solver="liblinear", max_iter=2000, random_state=SEED))]),
    "Random Forest": Pipeline([
        ("clipper", QuantileClipper()), ("selector", make_selector()),
        ("model", RandomForestClassifier(n_estimators=500, n_jobs=1, random_state=SEED))]),
    "XGBoost": Pipeline([
        ("clipper", QuantileClipper()), ("selector", make_selector()),
        ("model", XGBClassifier(n_estimators=300, objective="binary:logistic",
                                eval_metric="logloss", tree_method="hist",
                                n_jobs=1, random_state=SEED))]),
}

parameter_grids = {
    "Logistic Regression": {
        "selector__max_features": [20, 30],
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"],
    },
    "Random Forest": {
        "selector__max_features": [20, 30],
        "model__max_depth": [3, 6],
        "model__min_samples_leaf": [5, 15],
        "model__max_features": ["sqrt", 0.5],
        "model__class_weight": ["balanced"],
    },
    "XGBoost": {
        "selector__max_features": [20, 30],
        "model__max_depth": [2, 3],
        "model__learning_rate": [0.03, 0.07],
        "model__subsample": [0.7, 1.0],
        "model__colsample_bytree": [0.8],
        "model__reg_lambda": [5.0],
    },
}


## Fit the three searches

This is the first computationally intensive cell and may take several minutes. The printed progress identifies which model is currently being fitted. Every candidate includes feature clipping and selection inside each fold.

In [ ]:
searches = {}
search_summary_rows = []
for name, pipeline in base_pipelines.items():
    print(f"Optimising {name} ...")
    started = time.perf_counter()
    search = GridSearchCV(
        pipeline, parameter_grids[name], scoring=SCORING, refit="roc_auc",
        cv=tscv, n_jobs=-1, verbose=1, return_train_score=False, error_score="raise")
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - started
    searches[name] = search
    best_i = search.best_index_
    search_summary_rows.append({
        "model": name, "cv_roc_auc": search.cv_results_["mean_test_roc_auc"][best_i],
        "cv_roc_auc_std": search.cv_results_["std_test_roc_auc"][best_i],
        "cv_balanced_accuracy": search.cv_results_["mean_test_balanced_accuracy"][best_i],
        "cv_log_loss": -search.cv_results_["mean_test_neg_log_loss"][best_i],
        "elapsed_seconds": elapsed, "best_parameters": search.best_params_})
    print(f"Finished {name}: CV AUC={search.best_score_:.4f}, time={elapsed:.1f}s")

search_summary = pd.DataFrame(search_summary_rows).set_index("model")
search_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(search_summary.index, search_summary["cv_roc_auc"],
            yerr=search_summary["cv_roc_auc_std"], fmt="o", capsize=5, color="#4c78a8")
ax.axhline(0.5, color="0.4", linestyle="--", linewidth=1)
ax.set(title="Base-learner time-series CV ROC-AUC", ylabel="Mean ROC-AUC", xlabel="")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

## Frozen base-model predictions on the blending sample

`GridSearchCV` refits each winning pipeline on all 711 training observations. Those fitted pipelines are now frozen and used to create three probability columns on the chronological blending sample. Metrics below describe the base predictions available to the future blender; they are not final test results.

In [ ]:
base_models = {name: search.best_estimator_ for name, search in searches.items()}
blend_probabilities = pd.DataFrame(index=X_blend.index)
for name, model in base_models.items():
    blend_probabilities[name] = model.predict_proba(X_blend)[:, 1]
blend_probabilities["Simple Average"] = blend_probabilities[list(base_models)].mean(axis=1)

def probability_metrics(y_true, probability, cutoff=0.5):
    prediction = (np.asarray(probability) >= cutoff).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, probability),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "brier_score": brier_score_loss(y_true, probability),
        "log_loss": log_loss(y_true, probability),
        "positive_prediction_rate": prediction.mean(),
    }

blend_base_metrics = pd.DataFrame({
    name: probability_metrics(y_blend, blend_probabilities[name])
    for name in blend_probabilities.columns
}).T.sort_values("roc_auc", ascending=False)
blend_base_metrics


In [ ]:
base_probability_correlation = blend_probabilities[list(base_models)].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(base_probability_correlation, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, ax=ax)
ax.set_title("Base-probability correlation on blending sample")
plt.tight_layout(); plt.show()
base_probability_correlation

### Stage 4 checkpoint

Review CV means and standard deviations before interpreting blending-sample performance. A model with mean AUC near 0.50 or high fold dispersion may still add ensemble diversity, but it should not be described as a strong standalone predictor. Stage 5 will train a regularised Logistic Regression blender, select the probability trading cutoff without accessing the final test set, and then freeze the complete specification.

# Stage 5 - Logistic blender and probability cutoff

The blender is an L2-regularised Logistic Regression using only the three frozen base probabilities. Its regularisation strength is selected through an expanding-window split inside the blending sample. To avoid choosing the trading cutoff from in-sample fitted probabilities, chronological out-of-fold (OOF) blender probabilities are then constructed.

Two strategies will later be reported: a pre-specified cutoff of 0.50 and a high-confidence cutoff selected from the OOF blending predictions. The final test set remains untouched throughout this stage.

In [ ]:
from sklearn.base import clone

meta_feature_names = list(base_models)
X_meta = blend_probabilities[meta_feature_names].copy()
y_meta = y_blend.copy()
meta_tscv = TimeSeriesSplit(n_splits=4, test_size=40, gap=PURGE_GAP)
meta_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(penalty="l2", solver="liblinear", max_iter=2000, random_state=SEED)),
])
meta_grid = {"model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
             "model__class_weight": [None, "balanced"]}
meta_search = GridSearchCV(meta_pipeline, meta_grid, scoring=SCORING, refit="roc_auc",
                           cv=meta_tscv, n_jobs=-1, return_train_score=False, error_score="raise")
meta_search.fit(X_meta, y_meta)
meta_best_i = meta_search.best_index_
meta_search_summary = pd.Series({
    "cv_roc_auc": meta_search.cv_results_["mean_test_roc_auc"][meta_best_i],
    "cv_roc_auc_std": meta_search.cv_results_["std_test_roc_auc"][meta_best_i],
    "cv_balanced_accuracy": meta_search.cv_results_["mean_test_balanced_accuracy"][meta_best_i],
    "cv_log_loss": -meta_search.cv_results_["mean_test_neg_log_loss"][meta_best_i],
    "best_parameters": meta_search.best_params_,
})
meta_search_summary.to_frame("value")

In [ ]:
meta_oof_probability = pd.Series(np.nan, index=X_meta.index, name="Blending OOF probability")
for fit_idx, validation_idx in meta_tscv.split(X_meta):
    fold_blender = clone(meta_search.best_estimator_)
    fold_blender.fit(X_meta.iloc[fit_idx], y_meta.iloc[fit_idx])
    meta_oof_probability.iloc[validation_idx] = fold_blender.predict_proba(X_meta.iloc[validation_idx])[:, 1]
meta_oof = pd.DataFrame({
    "probability": meta_oof_probability,
    "target": y_meta,
    "target_return": blend_df["target_return"],
}).dropna()
print(f"Chronological meta-model OOF observations: {len(meta_oof)}")
print(f"OOF ROC-AUC: {roc_auc_score(meta_oof.target, meta_oof.probability):.4f}")

## Select the high-confidence trading cutoff

### Trading threshold selection

Candidate probability thresholds are set in advance between 0.50 and 0.65. For each threshold, the model takes a long position in AZN when the predicted probability exceeds the threshold and otherwise remains in cash.

A one-way transaction cost of 5 basis points is applied whenever the position changes. The transaction cost on day $t$ is calculated as:

$$
\text{Cost}_t = \frac{5}{10{,}000}\left|w_t-w_{t-1}\right|
$$

where $w_t$ is the position on day $t$, with 1 representing a long position and 0 representing cash.

The threshold is selected using the highest annualised net Sharpe ratio. To avoid choosing a strategy that appears successful only because it trades very rarely, the selected strategy must have at least 10 position changes and market exposure between 10% and 90%.

In [ ]:
def cutoff_strategy_metrics(frame, cutoff, one_way_cost_bps=5.0):
    position = (frame["probability"] >= cutoff).astype(float)
    turnover = position.diff().abs()
    turnover.iloc[0] = position.iloc[0]
    gross_return = position * frame["target_return"]
    net_return = gross_return - turnover * one_way_cost_bps / 10_000
    net_std = net_return.std(ddof=1)
    prediction = position.astype(int)
    return {
        "cutoff": cutoff,
        "balanced_accuracy": balanced_accuracy_score(frame["target"], prediction),
        "annualised_net_return": net_return.mean() * 252,
        "annualised_net_volatility": net_std * np.sqrt(252),
        "net_sharpe": np.sqrt(252) * net_return.mean() / net_std if net_std > 0 else np.nan,
        "exposure": position.mean(),
        "position_changes": int(turnover.sum()),
        "total_cost": (turnover * one_way_cost_bps / 10_000).sum(),
    }

CUTOFF_CANDIDATES = np.round(np.arange(0.50, 0.651, 0.025), 3)
cutoff_results = pd.DataFrame([cutoff_strategy_metrics(meta_oof, c, ONE_WAY_COST_BPS)
                               for c in CUTOFF_CANDIDATES]).set_index("cutoff")
eligible_cutoffs = cutoff_results.query("position_changes >= 10 and exposure >= 0.10 and exposure <= 0.90")
if eligible_cutoffs.empty:
    SELECTED_CUTOFF = 0.50
    print("No high-confidence cutoff passed the activity constraints; using 0.50.")
else:
    SELECTED_CUTOFF = float(eligible_cutoffs["net_sharpe"].idxmax())
print(f"Selected OOF high-confidence cutoff: {SELECTED_CUTOFF:.3f}")
cutoff_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cutoff_results["net_sharpe"].plot(marker="o", ax=axes[0], color="#4c78a8")
axes[0].axvline(SELECTED_CUTOFF, color="#e45756", linestyle="--")
axes[0].set(title="OOF net Sharpe by probability cutoff", ylabel="Annualised net Sharpe")
cutoff_results["exposure"].plot(marker="o", ax=axes[1], color="#72b7b2")
axes[1].axvline(SELECTED_CUTOFF, color="#e45756", linestyle="--")
axes[1].set(title="OOF exposure by probability cutoff", ylabel="Fraction long")
for ax in axes: ax.set_xlabel("Probability cutoff")
plt.tight_layout(); plt.show()

In [ ]:
# GridSearchCV has refitted this blender on the complete blending sample.
blender = meta_search.best_estimator_
standardised_coefficients = pd.Series(
    blender.named_steps["model"].coef_[0], index=meta_feature_names,
    name="standardised_meta_coefficient")
frozen_specification = pd.Series({
    "base_models": meta_feature_names,
    "blender": "L2 Logistic Regression",
    "standard_cutoff": 0.50,
    "selected_high_confidence_cutoff": SELECTED_CUTOFF,
    "one_way_cost_bps": ONE_WAY_COST_BPS,
    "final_test_used": False,
})
display(standardised_coefficients.to_frame())
frozen_specification.to_frame("value")

### Stage 5 checkpoint

The base models, blender, standard cutoff, high-confidence cutoff and transaction-cost assumption are now frozen. Stage 6 may evaluate the final test set exactly once. No hyperparameter or cutoff will be changed in response to the final-test results.

# Stage 6 - Final untouched test evaluation and P&L backtest

The full specification was frozen before this section. The final test sample is now evaluated without changing features, hyperparameters, ensemble weights, cutoffs or costs. Re-running the notebook reproduces the same fixed evaluation; it does not authorise selecting a new specification from these results.

The validation-selected cutoff of 0.525 had negative net Sharpe in blending OOF data. It is retained as a pre-specified sensitivity case, not described as a validated profitable rule. The 0.50 cutoff remains the primary classification strategy.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix
from src.evaluation import (classification_metrics, strategy_returns,
                            performance_metrics, bootstrap_auc_interval)

test_probabilities = pd.DataFrame(index=X_test.index)
for name, model in base_models.items():
    test_probabilities[name] = model.predict_proba(X_test)[:, 1]
test_probabilities["Simple Average"] = test_probabilities[meta_feature_names].mean(axis=1)
test_probabilities["Blending Ensemble"] = blender.predict_proba(test_probabilities[meta_feature_names])[:, 1]
assert test_probabilities.notna().all().all()
print(f"Final test observations evaluated under frozen specification: {len(test_probabilities)}")

In [ ]:
final_classification_metrics = pd.DataFrame({
    name: classification_metrics(y_test, test_probabilities[name], cutoff=0.50)
    for name in test_probabilities.columns
}).T.sort_values("roc_auc", ascending=False)
auc_intervals = {}
for name in test_probabilities.columns:
    low, high = bootstrap_auc_interval(y_test, test_probabilities[name], random_state=SEED)
    auc_intervals[name] = {"auc_ci_low": low, "auc_ci_high": high}
auc_intervals = pd.DataFrame(auc_intervals).T
final_classification_metrics = final_classification_metrics.join(auc_intervals)
final_classification_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name in test_probabilities.columns:
    RocCurveDisplay.from_predictions(y_test, test_probabilities[name], name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="0.5")
ax.set_title("Final test ROC curves")
plt.tight_layout(); plt.show()

In [ ]:
models_for_confusion = ["Logistic Regression", "Random Forest", "XGBoost",
                        "Simple Average", "Blending Ensemble"]
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, name in zip(axes.flat, models_for_confusion):
    pred = (test_probabilities[name] >= 0.50).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=[0, 1]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
axes.flat[-1].axis("off")
plt.suptitle("Final test confusion matrices at cutoff 0.50", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for name in ["Logistic Regression", "Random Forest", "XGBoost", "Blending Ensemble"]:
    observed, predicted = calibration_curve(y_test, test_probabilities[name], n_bins=6, strategy="quantile")
    axes[0].plot(predicted, observed, marker="o", label=name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="0.5")
axes[0].set(title="Final test calibration", xlabel="Mean predicted probability", ylabel="Observed positive rate")
axes[0].legend(fontsize=8)
sns.histplot(test_probabilities["Blending Ensemble"], bins=20, ax=axes[1], color="#4c78a8")
axes[1].axvline(0.50, color="#e45756", linestyle="--", label="0.50 cutoff")
axes[1].axvline(SELECTED_CUTOFF, color="#f2cf5b", linestyle=":", label=f"Selected {SELECTED_CUTOFF:.3f}")
axes[1].set(title="Blender probability distribution", xlabel="Predicted probability")
axes[1].legend()
plt.tight_layout(); plt.show()

## Final-test long/cash backtest

Each signal generated after close \(t\) controls exposure from open \(t+1\) to open \(t+2\). Costs are deducted when the desired position changes. Buy-and-hold is evaluated on the same adjusted open-to-open return observations and is charged the initial entry cost.

In [ ]:
test_asset_return = test_df["target_return"]
strategy_results = {
    "Buy and Hold AZN": strategy_returns(test_asset_return, always_long=True, one_way_cost_bps=ONE_WAY_COST_BPS),
}
for name in test_probabilities.columns:
    strategy_results[f"{name} (0.50)"] = strategy_returns(
        test_asset_return, test_probabilities[name], cutoff=0.50, one_way_cost_bps=ONE_WAY_COST_BPS)
if not np.isclose(SELECTED_CUTOFF, 0.50):
    strategy_results[f"Blender ({SELECTED_CUTOFF:.3f}, validation-selected)"] = strategy_returns(
        test_asset_return, test_probabilities["Blending Ensemble"], cutoff=SELECTED_CUTOFF,
        one_way_cost_bps=ONE_WAY_COST_BPS)

final_backtest_metrics = pd.DataFrame({
    name: performance_metrics(result) for name, result in strategy_results.items()
}).T.sort_values("sharpe", ascending=False)
final_backtest_metrics

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
equity_curves = {}
for name, result in strategy_results.items():
    if name in ["Buy and Hold AZN", "Simple Average (0.50)", "Blending Ensemble (0.50)",
                f"Blender ({SELECTED_CUTOFF:.3f}, validation-selected)"]:
        wealth = (1 + result["net_return"]).cumprod()
        equity_curves[name] = wealth
        axes[0].plot(wealth.index, wealth, label=name)
        axes[1].plot(wealth.index, wealth / wealth.cummax() - 1, label=name)
axes[0].set(title="Final test net equity curves", ylabel="Growth of $1")
axes[1].set(title="Final test drawdowns", ylabel="Drawdown", xlabel="Signal date")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Transaction-cost stress test

The model and cutoff remain fixed. Only the assumed one-way cost changes, so this is a genuine sensitivity analysis rather than test-set retuning.

In [ ]:
cost_stress_rows = []
for cost_bps in [0, 5, 10, 20]:
    result = strategy_returns(test_asset_return, test_probabilities["Blending Ensemble"],
                              cutoff=0.50, one_way_cost_bps=cost_bps)
    row = performance_metrics(result)
    row["one_way_cost_bps"] = cost_bps
    cost_stress_rows.append(row)
cost_stress = pd.DataFrame(cost_stress_rows).set_index("one_way_cost_bps")
cost_stress[["cumulative_return", "annualised_return", "sharpe",
             "maximum_drawdown", "total_transaction_cost"]]

### Stage 6 checkpoint

Interpret classification and economic performance separately. AUC measures ranking, while the backtest depends on calibration, cutoff, exposure, turnover and costs. If confidence intervals include 0.50 or the strategy underperforms buy-and-hold, this must be reported directly rather than repaired through final-test retuning.

# Stage 7 - Robustness, diagnostics, and reproducible outputs

These checks do not alter the frozen models. They examine whether the conclusions depend on the definition of a positive return, market regime, calendar subperiod, immediate execution horizon or transaction-cost assumption. Results are diagnostic rather than a new round of model selection.

## Near-zero return label sensitivity

The models were trained with \(\theta_r=0\). Without retraining, their frozen probability rankings are compared with stricter final-test labels of 5, 10 and 25 basis points. This asks whether a model trained on return sign also ranks economically larger positive moves. It is not presented as optimisation under the alternative labels.

In [ ]:
label_sensitivity_rows = []
for return_cutoff_bps in [0, 5, 10, 25]:
    alternative_y = (test_df["target_return"] > return_cutoff_bps / 10_000).astype(int)
    for name in ["XGBoost", "Simple Average", "Blending Ensemble"]:
        label_sensitivity_rows.append({
            "return_cutoff_bps": return_cutoff_bps, "model": name,
            "positive_rate": alternative_y.mean(),
            "roc_auc": roc_auc_score(alternative_y, test_probabilities[name]),
            "balanced_accuracy_at_0.50": balanced_accuracy_score(
                alternative_y, (test_probabilities[name] >= 0.50).astype(int))})
label_sensitivity = pd.DataFrame(label_sensitivity_rows).set_index(["return_cutoff_bps", "model"])
label_sensitivity

## Volatility and market-trend regimes

The high/low volatility boundary is the median AZN 20-day realised volatility from the base-training sample, not the final test sample. The market regime is defined by the sign of SPY's 20-day return. Subsample AUC is reported only when both classes are present.

In [ ]:
training_vol_boundary = train_df["azn_realized_vol_20"].median()
regime_masks = {
    "Low AZN volatility": test_df["azn_realized_vol_20"] <= training_vol_boundary,
    "High AZN volatility": test_df["azn_realized_vol_20"] > training_vol_boundary,
    "Positive SPY 20D trend": test_df["spy_return_20d"] > 0,
    "Non-positive SPY 20D trend": test_df["spy_return_20d"] <= 0,
}
regime_rows = []
for regime, mask in regime_masks.items():
    y_sub = y_test.loc[mask]
    for name in ["XGBoost", "Blending Ensemble"]:
        p_sub = test_probabilities.loc[mask, name]
        regime_rows.append({
            "regime": regime, "model": name, "observations": len(y_sub),
            "positive_rate": y_sub.mean(),
            "roc_auc": roc_auc_score(y_sub, p_sub) if y_sub.nunique() == 2 else np.nan,
            "balanced_accuracy": balanced_accuracy_score(y_sub, (p_sub >= 0.50).astype(int))})
regime_metrics = pd.DataFrame(regime_rows).set_index(["regime", "model"])
regime_metrics

## Quarterly stability

A one-year test period is too short for annual stability analysis, so results are grouped by calendar quarter. Each quarter contains relatively few observations; these figures are descriptive and should not be treated as independent significance tests.

In [ ]:
quarter_rows = []
blender_test_result = strategy_results["Blending Ensemble (0.50)"]
for quarter, idx in test_df.groupby(test_df.index.to_period("Q")).groups.items():
    idx = pd.Index(idx)
    y_q = y_test.loc[idx]
    p_q = test_probabilities.loc[idx, "Blending Ensemble"]
    result_q = blender_test_result.loc[idx]
    perf_q = performance_metrics(result_q)
    quarter_rows.append({
        "quarter": str(quarter), "observations": len(idx),
        "positive_rate": y_q.mean(),
        "roc_auc": roc_auc_score(y_q, p_q) if y_q.nunique() == 2 else np.nan,
        "balanced_accuracy": balanced_accuracy_score(y_q, (p_q >= 0.50).astype(int)),
        "strategy_return": perf_q["cumulative_return"],
        "strategy_sharpe": perf_q["sharpe"],
        "exposure": perf_q["exposure"]})
quarterly_stability = pd.DataFrame(quarter_rows).set_index("quarter")
quarterly_stability

## Additional one-day execution-delay sanity check

The fixed 0.50 blender position is shifted by one further trading day while returns remain unchanged. This deliberately misaligns the model from its intended horizon and tests whether any apparent value persists beyond the forecast window.

In [ ]:
normal_result = strategy_results["Blending Ensemble (0.50)"]
delayed_position = normal_result["position"].shift(1).fillna(0.0)
delayed_turnover = delayed_position.diff().abs()
delayed_turnover.iloc[0] = delayed_position.iloc[0]
delayed_result = pd.DataFrame(index=test_df.index)
delayed_result["asset_return"] = test_asset_return
delayed_result["position"] = delayed_position
delayed_result["turnover"] = delayed_turnover
delayed_result["cost"] = delayed_turnover * ONE_WAY_COST_BPS / 10_000
delayed_result["gross_return"] = delayed_position * test_asset_return
delayed_result["net_return"] = delayed_result["gross_return"] - delayed_result["cost"]
execution_delay_comparison = pd.DataFrame({
    "Intended horizon": performance_metrics(normal_result),
    "Additional one-day delay": performance_metrics(delayed_result),
}).T
execution_delay_comparison

## Majority-class and cash baselines

Because 57.14% of final-test observations are positive, a classifier that always predicts class 1 achieves 57.14% ordinary accuracy but exactly 50% balanced accuracy. This demonstrates why ordinary accuracy is not a primary metric. Cash has zero return and volatility and is included conceptually rather than ranked by Sharpe.

In [ ]:
majority_prediction = np.ones(len(y_test), dtype=int)
baseline_summary = pd.Series({
    "test_positive_rate": y_test.mean(),
    "majority_classifier_accuracy": (majority_prediction == y_test.to_numpy()).mean(),
    "majority_classifier_balanced_accuracy": balanced_accuracy_score(y_test, majority_prediction),
    "cash_cumulative_return": 0.0,
    "cash_exposure": 0.0,
})
baseline_summary.to_frame("value")

## Numerical implementation inventory

The project brief requests explicit disclosure of which numerical methods were implemented, modified or called from libraries.

In [ ]:
implementation_inventory = pd.DataFrame([
    ["Adjusted market-data download", "Library call with validation wrapper", "yfinance / own validation", "Acquire reproducible OHLCV inputs"],
    ["Technical indicators", "Library", "ta", "Established RSI, MACD, ATR, ADX, Bollinger and OBV calculations"],
    ["Additional feature generation", "Own functions", "src/features.py", "Returns, volatility estimators, interactions and external features"],
    ["Purged chronological splitting", "Modified implementation", "scikit-learn / own split logic", "Prevent target overlap and temporal leakage"],
    ["Quantile clipping", "Own transformer", "src/preprocessing.py", "Training-only extreme-value treatment"],
    ["Correlation and MI selection", "Own transformer using library MI", "scikit-learn / own grouping", "Reduce redundant predictors inside each CV fold"],
    ["Base learner fitting", "Library", "scikit-learn / XGBoost", "Estimate three complementary classifiers"],
    ["Blending logic", "Own pipeline", "scikit-learn / own OOF logic", "Combine frozen base probabilities without test leakage"],
    ["Backtest and P&L metrics", "Own functions", "src/evaluation.py", "Apply signals, turnover costs and performance measures"],
    ["AUC confidence intervals", "Own wrapper", "scikit-learn / NumPy bootstrap", "Quantify finite-sample uncertainty"],
], columns=["Technique", "Implementation", "Library / own code", "Purpose"])
implementation_inventory

## Empirical conclusion

The strongest final-test classifier was XGBoost, but its ROC-AUC of approximately 0.53 was not statistically distinguishable from 0.50 under the bootstrap interval. The blending ensemble did not improve ranking relative to the best base learner or simple averaging. Its long/cash strategy generated a positive return after assumed costs and reduced drawdown through partial exposure, but it underperformed buy-and-hold AZN in both total return and Sharpe ratio during a strongly positive test year. The evidence therefore supports, at most, weak and unstable predictability rather than a reliable trading edge.

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
final_classification_metrics.to_csv(RESULTS_DIR / "final_classification_metrics.csv")
final_backtest_metrics.to_csv(RESULTS_DIR / "final_backtest_metrics.csv")
cost_stress.to_csv(RESULTS_DIR / "transaction_cost_stress.csv")
label_sensitivity.to_csv(RESULTS_DIR / "label_sensitivity.csv")
regime_metrics.to_csv(RESULTS_DIR / "regime_metrics.csv")
quarterly_stability.to_csv(RESULTS_DIR / "quarterly_stability.csv")
implementation_inventory.to_csv(RESULTS_DIR / "implementation_inventory.csv", index=False)
print(f"Result tables saved to: {RESULTS_DIR.resolve()}")